# **Chess outcome prediction**

### Imports
These libraries are used throughout this jupyter notebook. 
Run the download_libraries.py script in the project folder in order to install all dependencies and libraries in your local environment

In [ ]:
import sklearn as sk
import pandas as pd
import numpy as np
import chess.pgn
import matplotlib.pyplot as plt
import seaborn

## **Loading data from the parquet files**

In [275]:
df_raw = pd.read_parquet("parsed")

## **Data preprocessing**

In [308]:
#Preprocessing 
df = df_raw
df["Event"] = df["Event"].astype(str)

df = df[(df["Event"]=="Rated Bullet game") | (df["Event"]=="Rated Blitz game")]
df = df[(df["WhiteElo"].astype(int) <= 1500) & (df["BlackElo"].astype(int) <=1500)]

for column in ["WhiteElo", "BlackElo"]:
    df[column] = pd.to_numeric(df[column], errors="coerce").astype("Int64")
df["ResultRaw"] = df["ResultRaw"].astype(str)
df["ECO"] = df["ECO"].astype(str)
df["Opening"] = df["Opening"].astype(str)

df["Datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"], format="%Y.%m.%d %H:%M:%S")

df["TimeControl"] = df["TimeControl"].astype(str)
df["Termination"] = df["Termination"].astype(str)


df = df.drop(columns=["Date", "Time", "Termination"])


## **Feature engineering**

### **Utility functions**
These are functions used on the dataframe in order extract features from the data.

In [309]:
def to_winprob(s):
    return s.white().wdl(model="lichess").expectation()

def transform_result(str_res):
    if str_res == "1-0":
        return 1.0
    elif str_res == "0-1":
        return -1.0
    else:
        return 0.0

def early_castle(san_list):
    if len(san_list) > 20:
        checkpoint = 20
    else:
        checkpoint = len(san_list)

    white = False
    black = False
    for i in range(0, checkpoint):
        if "O-O" in san_list[i]:
            if i % 2 == 0:
                white = True
            else:
                black = True

    return white, black

def early_queen(san_list):
    if len(san_list) > 20:
        checkpoint = 20
    else:
        checkpoint = len(san_list)

    white = False
    black = False
    for i in range(0, checkpoint):
        if "Q" in san_list[i]:
            if i % 2 == 0:
                white = True
            else:
                black = True

    return white, black

# def early_advantage(win_prob):
    if all(x is None for x in win_prob[:20]):
        return None

    if len(win_prob) > 20:
        checkpoint = 20
    else:
        checkpoint = len(win_prob)

    early_advantage = False
    for i in range(0, checkpoint):
        if win_prob[i] is not None and win_prob[i] > 0.55:
            early_advantage = True

    return early_advantage


def early_wp_mean(win_prob):
    values = []
    for x in win_prob[:20]:
        values.append(x)
    
    return sum(values) / len(values)

def early_double_move(san_list):
    if len(san_list) > 10:
        checkpoint = 10
    else:
        checkpoint = len(san_list)
    
    board = chess.Board()
    moved_white = set()
    moved_black = set()
    white = False
    black = False

    for i in range(0, checkpoint):
        move = board.push_san(san_list[i])

        if i % 2 == 0:
            if move.from_square in moved_white:
                white = True
            moved_white.discard(move.from_square)
            moved_white.add(move.to_square)
            moved_black.discard(move.to_square)
        else:
            if move.from_square in moved_black:
                black = True
            moved_black.discard(move.from_square)
            moved_black.add(move.to_square)
            moved_white.discard(move.to_square)

    return white, black

def parse_tc(tc):
    base, inc = tc.split("+")
    return int(base), int(inc)

def clock_features(row):
    base, inc = parse_tc(row["TimeControl"])
    clocks = row["Clocks"]
    if len(clocks) < 20 or clocks[18] is None or clocks[19] is None:
        return None, None, None

    white_left = clocks[18]
    black_left = clocks[19]

    white_used = (base + 10 * inc - white_left) / base
    black_used = (base + 10 * inc - black_left) / base
    clock_diff = (white_left - black_left) / base

    return white_used, black_used, clock_diff

### **Feature extraction**

In [ ]:
parts = df["Opening"].str.split(":", n=1)
df["OpeningFamily"] = parts.str[0].str.strip()
df["OpeningVariation"] = parts.str[1].str.strip() 

df["Result"] = df["ResultRaw"].apply(transform_result)
df["EarlyWPMean"] = df["WinProb"].apply(wp_mean)
df["EarlyAdvantage "]
df["EloDiff"] = df["WhiteElo"]-df["BlackElo"]
df["AvgElo"] = (df["WhiteElo"] + df["BlackElo"]) / 2


df[["WhiteEarlyCastle", "BlackEarlyCastle"]] = pd.DataFrame(df["Sans"].apply(early_castle).tolist(), index=df.index)
df[["WhiteEarlyQueen", "BlackEarlyQueen"]] = pd.DataFrame(df["Sans"].apply(early_queen).tolist(), index=df.index)
df[["WhiteEarlyDoubleMove", "BlackEarlyDoubleMove"]] = pd.DataFrame(df["Sans"].apply(early_double_move).tolist(), index=df.index)
df[["WhiteTimeUsed", "BlackTimeUsed", "ClockDiff"]] = pd.DataFrame(df.apply(clock_features, axis=1).tolist(), index=df.index)

top = df["OpeningFamily"].value_counts().head(20).index
df["OpeningGroup"] = df["OpeningFamily"].where(df["OpeningFamily"].isin(top), "Other")

df[["BaseTime", "Increment"]] = df["TimeControl"].str.split("+", expand=True).astype(int)
df = df.drop(columns=["TimeControl"])

df["EstDuration"] = df["BaseTime"] + 40 * df["Increment"] # the Lichess formula for diffferentiating blitz from bullet games.


df = df.drop(["Opening", "ResultRaw", "OpeningVariation", "WhiteElo", "BlackElo", "OpeningFamily", "OpeningVariation", "ECO"], axis=1)

KeyError: 'EarlyAdvantage '

## **Data visualisation**

In [306]:
df.head()

,Event,Sans,WinProb,Clocks,Datetime,Result,EarlyAdvantage,EloDiff,AvgElo,WhiteEarlyCastle,...,BlackEarlyQueen,WhiteEarlyDoubleMove,BlackEarlyDoubleMove,WhiteTimeUsed,BlackTimeUsed,ClockDiff,OpeningGroup,BaseTime,Increment,EstDuration
0,Rated Bullet game,"[e4, c5, Bc4, e6, Nf3, a6, Nc3, Nc6, d3, g6, O...","[0.503, 0.517, 0.494, 0.495, 0.493, 0.507, 0.4...","[60.0, 60.0, 58.0, 59.0, 57.0, 59.0, 55.0, 57....",2018-05-31 22:00:13,1.0,0.49845,-67,1122.5,True,...,False,False,False,0.183333,0.216667,0.033333,Sicilian Defense,60,0,60
2,Rated Blitz game,"[e4, e5, Nf3, Nc6, Bb5, Nf6, d3, Bc5, h3, O-O,...","[0.503, 0.534, 0.523, 0.519, 0.513, 0.524, 0.5...","[300.0, 300.0, 301.0, 302.0, 302.0, 302.0, 300...",2018-05-31 22:00:15,-1.0,0.47255,-2,1223.0,True,...,True,False,False,0.266667,0.250000,-0.016667,Ruy Lopez,300,3,420
5,Rated Bullet game,"[e4, d6, d4, g6, c4, Bg7, Nc3, Nf6, e5, Nbd7, ...","[0.503, 0.523, 0.535, 0.527, 0.529, 0.534, 0.5...","[60.0, 60.0, 58.0, 60.0, 55.0, 59.0, 53.0, 59....",2018-05-31 22:00:19,-1.0,0.68925,-42,1024.0,False,...,False,True,False,0.350000,0.166667,-0.183333,Other,60,0,60
11,Rated Bullet game,"[e3, e5, d3, d5, Nf3, Bb4+, Bd2, Bxd2+, Qxd2, ...","[0.5, 0.511, 0.5, 0.486, 0.482, 0.493, 0.483, ...","[60.0, 60.0, 60.0, 60.0, 59.0, 59.0, 58.0, 57....",2018-05-31 22:00:08,1.0,0.52365,55,1151.5,True,...,True,False,True,0.066667,0.116667,0.050000,Van't Kruijs Opening,60,0,60
14,Rated Blitz game,"[d4, e6, e3, g6, Nf3, Bg7, g3, f5, Bg2, Nf6, N...","[0.507, 0.518, 0.506, 0.543, 0.53, 0.535, 0.48...","[180.0, 180.0, 181.0, 182.0, 182.0, 183.0, 183...",2018-05-31 22:00:12,1.0,0.51900,64,1408.0,False,...,True,False,False,0.105556,0.044444,-0.061111,Other,180,2,260
